# 20차시 실습 — 내 서비스를 '평가'하기 (Langfuse 평가)

> **day1 → day2 → day3 연속 실습.** 19차시에서 프롬프트를 자산으로 관리했다면, 이제 **"바꾼 게 정말 더 나은지"** 를 숫자로 확인합니다.
> 평가 데이터셋 + 평가자(규칙·LLM-as-judge)로 **점수(Score)** 를 매기고, 버전을 **실험(Experiment)** 으로 비교합니다.
> 로컬로 개념을 잡고, 같은 것을 **자체 호스팅 Langfuse(Docker)** 의 Datasets·Experiments·Scores로 올립니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- **Langfuse 키가 없어도 STEP 1~3(로컬 평가)로 전부 실행됩니다.** STEP 4는 자체 호스팅 Langfuse가 있을 때 동작합니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 순서대로 실행하세요(`Shift+Enter`).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

18차시: 운영 중 **무슨 일이 있었는지** 관찰(trace·score)했습니다.
19차시: 프롬프트를 **버전 자산**으로 바꿨습니다.
20차시: "그 새 버전이 **정말 더 좋은가?**" — 느낌이 아니라 **평가 데이터셋 + 점수**로 답합니다.

> 공통 예시 = **맛집 추천 도우미**. 구조는 동일하니 **당신 팀 MVP**로 바꾸면 그대로 적용됩니다(STEP 7).
> 핵심: 회귀(품질 저하)를 **사용자보다 먼저** 발견하려면, 재현 가능한 **평가 셋 + 자동 채점**이 필요하다.

## STEP 1 — 평가 데이터셋 만들기 (input + 기대값)

평가의 출발은 **대표 입력 모음**입니다. 각 항목은 `input`(사용자 요청)과, 있으면 `expected`(기대 정답/근거)를 가집니다.
정답이 하나로 안 떨어지는 생성 작업이 많으니, **기대값은 '반드시 포함할 핵심'** 수준으로 느슨하게 둡니다.

⌨️ **터미널 Codex:** `codex "맛집 추천 도우미 평가용 데이터셋을 input과 기대 키워드로 5개 만들어줘"`

In [2]:
# 평가 데이터셋 — 대표 입력 + 기대 핵심(있으면). 실제로는 로그/사용자 피드백에서 모읍니다.
DATASET = [
    {"input": "강남에서 2만원 이하 매운 점심 추천해줘.",       "expected": "강남"},
    {"input": "혼밥하기 좋은 국밥집 한 곳만 추천해줘.",        "expected": "국밥"},
    {"input": "홍대 근처 데이트하기 좋은 분위기 식당 알려줘.",  "expected": "홍대"},
    {"input": "예산 1만원으로 아침 먹을 곳?",                  "expected": None},
    {"input": "매운 거 못 먹는데 순한 점심 추천.",            "expected": None},
]
print(f"평가 항목 {len(DATASET)}개")

평가 항목 5개


## STEP 2 — 평가자(evaluator): 규칙 기반 + LLM-as-a-judge

점수를 매기는 **함수**를 만듭니다. 두 종류를 씁니다.
- **규칙 기반**: 싸고 빠르고 결정적 — 형식/키워드/길이 같은 '지켜야 할 것'을 검사.
- **LLM-as-a-judge**: 사람 평가에 가깝게 — 다른 LLM이 **정확성·관련성·간결성**을 0~1로 채점(이유 포함).

⌨️ **터미널 Codex:** `codex "답변을 0~1로 채점하는 규칙 기반 평가자와 LLM 심판 평가자를 만들어줘"`

In [3]:
# (a) 평가 대상 앱 — 맛집 추천 (19차시 프롬프트를 단순화해 인라인)
SYSTEM = "너는 맛집 추천 도우미다. 사용자 요청에 한국어로 간결히, 한 곳만 추천하라."
def app(user_input, temperature=0.7):
    resp = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": user_input}])
    return resp.choices[0].message.content.strip()

# (b) 규칙 기반 평가자 → (점수 0~1, 이유)
def rule_eval(item, output):
    score, why = 1.0, []
    if not output:
        return 0.0, "빈 응답"
    if item.get("expected") and item["expected"] not in output:
        score -= 0.5; why.append(f"기대 키워드 '{item['expected']}' 없음")
    if len(output) > 400:
        score -= 0.25; why.append("너무 김(>400자)")
    return round(max(score, 0.0), 2), ", ".join(why) or "형식 OK"

# (c) LLM-as-a-judge 평가자 → (점수 0~1, 이유). JSON 강제 출력.
def judge_eval(item, output):
    rubric = "정확성(요청 조건 반영)·관련성(맛집 추천다움)·간결성 관점에서 0~1 점수."
    resp = client.chat.completions.create(
        model=MODEL, temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": "너는 엄격한 평가자다. JSON만 출력한다."},
                  {"role": "user", "content":
                   f"[요청]\n{item['input']}\n\n[답변]\n{output}\n\n[기준] {rubric}\n"
                   '형식: {"score": 0~1 사이 float, "reason": "한 줄 이유"}'}])
    d = json.loads(resp.choices[0].message.content)
    return round(float(d["score"]), 2), d.get("reason", "")

# 빠른 확인
o = app(DATASET[0]["input"])
print("답변:", o[:70], "...")
print("규칙:", rule_eval(DATASET[0], o))
print("심판:", judge_eval(DATASET[0], o))

답변: 강남의 '마라탕'이 유명한 '하오마라탕'을 추천합니다. 매운 국물과 다양한 재료를 선택할 수 있어 맛있는 점심을 즐기기 좋습니 ...
규칙: (1.0, '형식 OK')
심판: (0.8, '요청 조건을 잘 반영했으나, 구체적인 가격 정보가 부족함.')


## STEP 3 — 로컬 실험(Experiment) + 두 버전 비교

데이터셋 × 평가자 = **실험 한 번**. 앱을 전 항목에 돌려 점수를 집계합니다(= Langfuse Experiment의 축소판).
그리고 **버전 A vs B**(예: temperature 0.2 vs 0.9)를 같은 셋으로 돌려 **평균 점수로 비교** — "어느 쪽이 더 나은가"를 숫자로.

⌨️ **터미널 Codex:** `codex "데이터셋에 앱을 돌려 규칙·심판 점수를 집계하고 두 설정을 비교하는 실험 루프를 만들어줘"`

In [4]:
def run_experiment(dataset, run_name, temperature=0.7, show=True):
    """데이터셋 전체에 앱을 돌리고 규칙·심판 점수를 집계 (로컬 Experiment)."""
    rows = []
    for it in dataset:
        out = app(it["input"], temperature=temperature)
        rs, _ = rule_eval(it, out)
        js, _ = judge_eval(it, out)
        rows.append({"input": it["input"][:22], "rule": rs, "judge": js})
    rule_avg = sum(r["rule"] for r in rows) / len(rows)
    judge_avg = sum(r["judge"] for r in rows) / len(rows)
    if show:
        print(f"=== 실험 '{run_name}' (temp={temperature}) ===")
        for r in rows:
            print(f"  rule={r['rule']:.2f} judge={r['judge']:.2f} · {r['input']}…")
        print(f"  ▶ 평균  rule={rule_avg:.2f}  judge={judge_avg:.2f}")
    return {"run": run_name, "rule_avg": round(rule_avg, 2), "judge_avg": round(judge_avg, 2)}

a = run_experiment(DATASET, "A·보수(temp0.2)", temperature=0.2)
print()
b = run_experiment(DATASET, "B·창의(temp0.9)", temperature=0.9)
print("\n📊 비교:", a, "vs", b)
print("→ 평균 점수가 높은 설정을 'production'으로 승격 (19차시 프롬프트 라벨과 연결).")

=== 실험 'A·보수(temp0.2)' (temp=0.2) ===
  rule=1.00 judge=0.80 · 강남에서 2만원 이하 매운 점심 추천해줘…
  rule=0.50 judge=0.90 · 혼밥하기 좋은 국밥집 한 곳만 추천해줘.…
  rule=1.00 judge=0.80 · 홍대 근처 데이트하기 좋은 분위기 식당 …
  rule=1.00 judge=0.80 · 예산 1만원으로 아침 먹을 곳?…
  rule=1.00 judge=0.50 · 매운 거 못 먹는데 순한 점심 추천.…
  ▶ 평균  rule=0.90  judge=0.76

=== 실험 'B·창의(temp0.9)' (temp=0.9) ===
  rule=1.00 judge=0.90 · 강남에서 2만원 이하 매운 점심 추천해줘…
  rule=1.00 judge=1.00 · 혼밥하기 좋은 국밥집 한 곳만 추천해줘.…
  rule=1.00 judge=0.80 · 홍대 근처 데이트하기 좋은 분위기 식당 …
  rule=1.00 judge=0.80 · 예산 1만원으로 아침 먹을 곳?…
  rule=1.00 judge=0.80 · 매운 거 못 먹는데 순한 점심 추천.…
  ▶ 평균  rule=1.00  judge=0.86

📊 비교: {'run': 'A·보수(temp0.2)', 'rule_avg': 0.9, 'judge_avg': 0.76} vs {'run': 'B·창의(temp0.9)', 'rule_avg': 1.0, 'judge_avg': 0.86}
→ 평균 점수가 높은 설정을 'production'으로 승격 (19차시 프롬프트 라벨과 연결).


## STEP 4 — Langfuse 평가 (자체 호스팅): Datasets · Experiments · Scores

로컬 실험을 **Langfuse**로 올리면: 데이터셋이 **버전 관리**되고, 실험 결과가 **런별로 저장·비교**되며,
점수가 **Scores 대시보드**에 추세로 쌓입니다. SDK: `create_dataset` → `create_dataset_item` → `dataset.run_experiment(task=..., evaluators=[...])`.

> **Langfuse 평가자 규칙:** 평가자 함수는 dict가 아니라 **`langfuse.Evaluation(name=, value=, comment=)` 객체**를 반환해야 합니다. 항목별 채점은 `evaluators=`, 런 전체 집계(평균 등)는 `run_evaluators=`로 넘깁니다.

> **이 캠프는 Langfuse를 자체 호스팅(Docker, `http://localhost:3000`)으로 씁니다** — 0차시에서 Docker 설치.
> 띄우기: `git clone https://github.com/langfuse/langfuse.git && cd langfuse && docker compose up -d` → `localhost:3000` 가입 → **Settings ▸ API Keys** 복사 → 환경변수 설정 후 재실행.
> 키가 없으면 이 셀은 건너뜁니다(STEP 1~3 로컬 평가로 개념은 이미 완성).

⌨️ **터미널 Codex:** `codex "자체 호스팅 Langfuse에 데이터셋을 만들고 run_experiment로 규칙·LLM심판 평가자를 붙여 실험을 돌려줘"`

In [5]:
# Langfuse 평가 — 자체 호스팅 + 키가 있을 때만 실행
print("Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):")
print("     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse")
print("     2) docker compose up -d            # http://localhost:3000 에서 열림")
print("     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사")
print("     4) 환경변수 설정 후 이 셀 재실행:")
print("        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000")
print("   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.")

# 자체 호스팅(Self-host): Docker로 직접 띄운 주소(0차시에서 Docker 설치). 기본 http://localhost:3000
from langfuse import observe, get_client, Evaluation
from langfuse.openai import openai 
os.environ["LANGFUSE_HOST"] = getpass.getpass("LANGFUSE_HOST (화면에 안 보임): ")
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass.getpass("LANGFUSE_PUBLIC_KEY (화면에 안 보임): ")
os.environ["LANGFUSE_SECRET_KEY"] = getpass.getpass("LANGFUSE_SECRET_KEY (화면에 안 보임): ")

os.environ["LANGFUSE_DEBUG"] = "true"
    
lf = get_client()

# 1) 데이터셋 + 아이템 등록 (한 번만 — 이미 있으면 예외를 무시)
DATASET_NAME = "matjip-eval"
try:
    lf.create_dataset(name=DATASET_NAME)
    for it in DATASET:
        lf.create_dataset_item(dataset_name=DATASET_NAME,
                                input=it["input"], expected_output=it.get("expected"))
except Exception as e:
    print("데이터셋 생성 건너뜀(이미 있을 수 있음):", type(e).__name__)

# 2) 평가자 — run_experiment 시그니처: (*, input, output, expected_output, metadata, **kw) → langfuse.Evaluation
#    ⚠️ dict가 아니라 Evaluation 객체를 반환해야 함 (SDK가 .name/.value 속성으로 접근 →
#       dict를 반환하면 "'dict' object has no attribute 'name'" 에러).
def rule_evaluator(*, input, output, expected_output=None, **kw):
    score, _ = rule_eval({"input": input, "expected": expected_output}, output)
    return Evaluation(name="rule_accuracy", value=score)
def judge_evaluator(*, input, output, expected_output=None, **kw):
    score, reason = judge_eval({"input": input, "expected": expected_output}, output)
    return Evaluation(name="llm_judge", value=score, comment=reason)

# 2-b) run-level 평가자 — 런 전체 항목의 점수를 집계 (Langfuse가 Runs 요약에 표시)
#      시그니처: (*, item_results, **kw) → langfuse.Evaluation
def avg_llm_judge(*, item_results, **kw):
    vals = [e.value for r in item_results for e in r.evaluations
            if e.name == "llm_judge" and isinstance(e.value, (int, float))]
    avg = round(sum(vals) / len(vals), 3) if vals else None
    return Evaluation(name="avg_llm_judge", value=avg,
                      comment=(f"평균 심판 점수 {avg}" if avg is not None else "점수 없음"))

# 3) task — 각 데이터셋 아이템을 앱에 통과 (반환값이 평가자의 output이 됨)
def task(*, item, **kw):
    return app(item.input)

ds = lf.get_dataset(DATASET_NAME)
result = ds.run_experiment(
    name="run-v1",
    description="맛집 추천 v1 — 규칙+LLM심판 평가",
    task=task,
    evaluators=[rule_evaluator, judge_evaluator],   # 항목별 평가자
    run_evaluators=[avg_llm_judge])                 # 런 전체 집계 평가자
lf.flush()
print("✅ 실험 완료 —", os.environ["LANGFUSE_HOST"], "→ Datasets ▸ matjip-eval ▸ Runs / Scores에서 확인")
try:
    print(result.format())          # 런 요약(SDK 버전에 따라 제공)
except Exception:
    pass

Langfuse를 '자체 호스팅(Self-host)'으로 띄웁니다 (0차시에서 Docker 설치):
     1) git clone https://github.com/langfuse/langfuse.git && cd langfuse
     2) docker compose up -d            # http://localhost:3000 에서 열림
     3) localhost:3000 가입(로컬 계정) → 프로젝트 생성 → Settings ▸ API Keys에서 키 2개 복사
     4) 환경변수 설정 후 이 셀 재실행:
        LANGFUSE_PUBLIC_KEY · LANGFUSE_SECRET_KEY · LANGFUSE_HOST=http://localhost:3000
   참고: from langfuse.openai import openai 로 바꾸면 토큰·비용까지 자동 계측됩니다.


2026-07-10 23:20:53,204 - langfuse - DEBUG - Thread: Media upload consumer thread #0 started and actively processing queue items
2026-07-10 23:20:53,204 - langfuse - DEBUG - Prompt cache initialized.
2026-07-10 23:20:53,205 - langfuse - DEBUG - Startup: Score ingestion consumer thread #0 started with batch size 15 and interval 1s
2026-07-10 23:20:53,206 - langfuse - INFO - Startup: Langfuse tracer successfully initialized | public_key=pk-lf-ecd84f7b-b38a-4be5-9458-a40eb8a03784 | base_url=http://localhost:3000 | environment=default | sample_rate=1.0 | media_threads=1
2026-07-10 23:20:53,206 - langfuse - DEBUG - Creating datasets matjip-eval
2026-07-10 23:20:53,390 - langfuse - DEBUG - Creating dataset item for dataset matjip-eval
2026-07-10 23:20:53,487 - langfuse - DEBUG - Creating dataset item for dataset matjip-eval
2026-07-10 23:20:53,502 - langfuse - DEBUG - Creating dataset item for dataset matjip-eval
2026-07-10 23:20:53,519 - langfuse - DEBUG - Creating dataset item for dataset 

✅ 실험 완료 — http://localhost:3000 → Datasets ▸ matjip-eval ▸ Runs / Scores에서 확인
Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: run-v1
📋 Run name: run-v1 - 2026-07-10T14:20:53.597618Z - 맛집 추천 v1 — 규칙+LLM심판 평가
5 items
Evaluations:
  • llm_judge
  • rule_accuracy

Average Scores:
  • llm_judge: 0.760
  • rule_accuracy: 1.000

Run Evaluations:
  • avg_llm_judge: 0.760
    💭 평균 심판 점수 0.76

🔗 Dataset Run:
   http://localhost:3000/project/cmrelzoev0006qo07uxnrg0em/datasets/cmrf0wqnx000aqo07c8azzt3j/runs/8505b65d-fec5-484f-9073-37b67342d966


## STEP 4-B — ⭐ Langfuse에 'Evaluator' 등록 (서버측 LLM-as-a-judge)

STEP 4는 **평가 로직을 코드로** 넘겼습니다(`evaluators=[...]`). 여기서는 반대로 **Langfuse에 Evaluator를 등록**해 두고,
실험을 돌리면 **서버가 자동으로 채점**하게 만듭니다 — 팀 전체가 같은 기준을 공유하고, 상시 트래픽에도 재사용됩니다.

- **Evaluator**: 어떻게 채점할지 (LLM-as-a-judge 프롬프트·모델·점수 이름)
- **Evaluation Rule**: 무엇을 채점할지 (데이터셋/트레이스 + 샘플링 + 변수 매핑)
- 등록 후 `dataset.run_experiment(name, task)` — **로컬 evaluators 없이** 서버 Evaluator가 채점

> API: `POST /api/public/unstable/evaluators`, `POST /api/public/unstable/evaluation-rules` (현재 **unstable**).
> **Evaluator를 만들려면 프로젝트에 LLM Connection이 최소 1개** 있어야 합니다 — 이 셀이 없으면 `OPENAI_API_KEY`로 자동 생성합니다(프로젝트 설정 ▸ LLM Connections에서도 확인 가능). 서버 채점은 **비동기**라 실행 직후가 아니라 잠시 뒤 Scores에 반영됩니다.
> STEP 4에서 넣은 키를 그대로 사용하며, 키가 없으면 이 셀은 건너뜁니다.

⌨️ **터미널 Codex:** `codex "Langfuse unstable evaluators API로 LLM-as-a-judge 평가자를 만들고 데이터셋 런에 연결해줘"`

In [13]:
# STEP 4-B — Langfuse에 'Evaluator(LLM-as-a-judge)'를 등록해 서버측 자동 채점
# 앞 STEP 4에서 설정한 LANGFUSE_HOST / PUBLIC_KEY / SECRET_KEY 를 그대로 사용합니다.
import requests

def _lf_cfg():
    host = os.environ.get("LANGFUSE_HOST", "").rstrip("/")
    pk, sk = os.environ.get("LANGFUSE_PUBLIC_KEY"), os.environ.get("LANGFUSE_SECRET_KEY")
    if not (host and pk and sk):
        raise RuntimeError("LANGFUSE_HOST/PUBLIC_KEY/SECRET_KEY 가 필요합니다 (STEP 4 셀 먼저 실행).")
    return host, (pk, sk)

def lf_api(method, path, payload=None):
    """Langfuse 공개 REST API 호출 (Basic Auth = public/secret 키)."""
    host, auth = _lf_cfg()
    r = requests.request(method, f"{host}{path}", auth=auth, json=payload, timeout=60)
    if not r.ok:                          # 검증 실패 시 서버가 알려주는 issue 전체 출력
        print(f"⚠️ {method} {path} -> {r.status_code}: {r.text}")
    r.raise_for_status()
    return r.json() if r.text else {}

EVALUATOR_NAME = "matjip-accuracy-judge"
try:
    host, _ = _lf_cfg()

    # 0-a) LLM Connection 보장 — Evaluator/서버 심판은 프로젝트에 'LLM 연결'이 최소 1개 필요
    #      없으면 STEP 1에서 넣은 OPENAI_API_KEY로 하나 만든다 (PUT = 업서트).
    try:
        conns = lf_api("GET", "/api/public/llm-connections")
        have = conns.get("data", conns) if isinstance(conns, dict) else conns
        if have:
            print("기존 LLM Connection:", [c.get("provider") for c in have][:5])
        else:
            openai_key = os.environ.get("OPENAI_API_KEY")
            if not openai_key:
                raise RuntimeError("LLM Connection이 없고 OPENAI_API_KEY도 없습니다 — STEP 1에서 키를 먼저 넣으세요.")
            lf_api("PUT", "/api/public/llm-connections", {
                "provider": "openai",          # 프로젝트 내 고유 이름(업서트 키)
                "adapter": "openai",           # openai / anthropic / azure / bedrock / ...
                "secretKey": openai_key,       # 서버가 이 키로 심판 LLM 호출
                "withDefaultModels": True,     # gpt-4o-mini 등 기본 모델 포함
            })
            
            print("✅ LLM Connection 생성: openai (기본 모델 포함) — 이제 Evaluator가 이 연결로 채점")
    except Exception as e:
        print("LLM Connection 확인/생성 응답:", type(e).__name__, "-", e)

    # 0) 이미 등록된 Evaluator 확인 (unstable API — GET으로 실제 응답 형태를 볼 수 있음)
    try:
        existing = lf_api("GET", "/api/public/unstable/evaluators")
        items = existing.get("data", existing) if isinstance(existing, dict) else existing
        print("등록된 Evaluator 수:", len(items) if hasattr(items, "__len__") else "?")
    except Exception:
        pass

    # 1) LLM-as-a-judge Evaluator 생성/버전업
    #    ⚠️ outputDefinition.dataType 는 필수 (NUMERIC / BOOLEAN / CATEGORICAL).
    #       NUMERIC/BOOLEAN 은 score.description + reasoning.description 을 준다.
    JUDGE_PROMPT = (
        "너는 맛집 추천 품질 심판이다. 사용자 요청과 답변, 기대 핵심을 보고 "
        "정확성·관련성·간결성을 종합해 0~1로 채점하라.\n\n"
        "요청: {{input}}\n답변: {{output}}\n기대 핵심: {{expected_output}}"
    )
    try:
        evaluator = lf_api("POST", "/api/public/unstable/evaluators", {
            "type": "llm_as_judge",
            "name": EVALUATOR_NAME,
            "prompt": JUDGE_PROMPT,
            "outputDefinition": {
                "dataType": "NUMERIC",
                "score":     {"description": "0~1 사이 정확성·관련성·간결성 종합 점수."},
                "reasoning": {"description": "그렇게 채점한 이유를 한국어 한 줄로."},
            },
            "modelConfig": {
                "provider": "openai",
                "model": "gpt-4o-mini"
            },
        })
        print("✅ Evaluator 등록:", evaluator.get("name", EVALUATOR_NAME),
              "· variables:", evaluator.get("variables"))
    except Exception as e:
        evaluator = {}
        print("Evaluator 생성 응답:", type(e).__name__, "— 이미 있으면 새 버전으로 처리됩니다.")

    # 2) Evaluation Rule — 이 Evaluator를 '데이터셋 실험(target=experiment)'에 자동 적용
    #    mapping: 프롬프트의 {{변수}} 각각을 실험 아이템 필드(source)에 1:1 연결
    SRC = {"input": "input", "output": "output", "expected_output": "expected_output"}
    variables = evaluator.get("variables") or list[str](SRC)          # 응답의 variables 우선
    mapping = [{"variable": v, "source": SRC.get(v, "input")} for v in variables]
    try:
        lf_api("POST", "/api/public/unstable/evaluation-rules", {
            "name": "matjip-accuracy-rule",
            "evaluator": {"name": EVALUATOR_NAME, "scope": "project", "type": "llm_as_judge"},
            "target": "experiment",         # 데이터셋 실험 런에 적용 (관찰 트레이스는 "observation")
            "enabled": True,
            "sampling": 1.0,                # 100% 샘플링
            "filter": [],                   # 모든 실험 런
            "mapping": mapping,
        })
        print("✅ Evaluation Rule 연결 — 실험 런을 서버가 자동 채점")
    except Exception as e:
        print("Rule 생성 응답:", type(e).__name__, "— 이미 연결돼 있을 수 있음(계속 진행).")

    # 3) 로컬 evaluators 없이 실험 실행 → Langfuse Evaluator가 자동 채점
    ds = lf.get_dataset(DATASET_NAME)
    ds.run_experiment(name="run-with-langfuse-judge",
                      description="서버측 LLM-as-a-judge Evaluator 자동 채점",
                      task=task)            # ← evaluators 인자 없음: 서버 Evaluator가 채점
    lf.flush()
    print("✅ 실험 완료 — UI ▸ Datasets ▸", DATASET_NAME,
          "▸ Runs 에서 'matjip-accuracy-judge' 점수를 확인하세요. (서버 채점은 비동기 — 잠시 후 반영)")
except Exception as e:
    print("건너뜀:", type(e).__name__, "-", e)
    print("· 자체 호스팅 Langfuse가 떠 있고 STEP 4에서 키를 넣었는지 확인하세요.")
    print("· 서버측 심판이 실제 채점하려면 프로젝트 설정 ▸ LLM Connections 에 LLM API 키가 있어야 합니다.")

기존 LLM Connection: ['openai']
등록된 Evaluator 수: 22
✅ Evaluator 등록: matjip-accuracy-judge · variables: ['input', 'output', 'expected_output']


2026-07-11 01:05:35,251 - langfuse - DEBUG - Getting datasets matjip-eval
2026-07-11 01:05:35,302 - langfuse - DEBUG - Starting experiment 'run-with-langfuse-judge' run 'run-with-langfuse-judge - 2026-07-10T16:05:35.300862Z' with 5 items
2026-07-11 01:05:35,383 - langfuse - DEBUG - Propagated 6 attributes to span '2491d9935c7fb69f': {'langfuse.experiment.id': '0acfe133-4abb-44ee-8004-7a68841e3ecf', 'langfuse.experiment.name': 'run-with-langfuse-judge - 2026-07-10T16:05:35.300862Z', 'langfuse.experiment.dataset.id': 'cmrf0wqnx000aqo07c8azzt3j', 'langfuse.experiment.item.id': 'bfc2b01b-a9a6-425c-9470-49c719af842c', 'langfuse.experiment.item.root_observation_id': 'fb7d0cea1c5aa187', 'langfuse.environment': 'sdk-experiment'}


✅ Evaluation Rule 연결 — 실험 런을 서버가 자동 채점


2026-07-11 01:05:36,703 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "56d21fb13504f65ab8dc986b256ab106",
    "span_id": "2491d9935c7fb69f",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": "fb7d0cea1c5aa187",
  "start_time": "2026-07-10T16:05:35.383612Z",
  "end_time": "2026-07-10T16:05:36.702582Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.experiment.id": "0acfe133-4abb-44ee-8004-7a68841e3ecf",
    "langfuse.experiment.name": "run-with-langfuse-judge - 2026-07-10T16:05:35.300862Z",
    "langfuse.experiment.dataset.id": "cmrf0wqnx000aqo07c8azzt3j",
    "langfuse.experiment.item.id": "bfc2b01b-a9a6-425c-9470-49c719af842c",
    "langfuse.experiment.item.root_observation_id": "fb7d0cea1c5aa187",
    "langfuse.environment": "sdk-experiment",
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub2

✅ 실험 완료 — UI ▸ Datasets ▸ matjip-eval ▸ Runs 에서 'matjip-accuracy-judge' 점수를 확인하세요. (서버 채점은 비동기 — 잠시 후 반영)


## STEP 4-C — 사용자·세션 추적 + 사람 주석(Human Annotation)

자동 평가(규칙·LLM 심판)만으로는 부족할 때가 있습니다. **누가(user)**·**어느 대화(session)** 에서 나온 답인지 묶어 보고,
사람이 직접 **👍/👎·등급**을 남기는 **HITL**을 더하면 평가가 완성됩니다.

- **user_id / session_id**: 트레이스에 붙이면 사용자별·세션별로 비용·품질을 집계·필터 (`propagate_attributes`)
- **사람 주석(Score)**: `create_score(trace_id=..., data_type="BOOLEAN"/"CATEGORICAL")` — 검수자의 👍/👎·등급
- **세션 점수**: `create_score(session_id=..., data_type="NUMERIC"/"BOOLEAN")` — 대화 전체 만족도
- **Annotation Queues**(UI): 대량 트레이스를 팀에 배분해 일관된 기준으로 라벨링

> 사람 점수는 LLM 심판 점수와 **같은 Scores 뷰**에 쌓여, 자동 vs 사람 평가를 나란히 비교할 수 있습니다.
> STEP 4(Langfuse 키)·STEP 1(OPENAI_API_KEY)을 먼저 실행해야 하며, 키가 없으면 이 셀은 건너뜁니다.

⌨️ **터미널 Codex:** `codex "트레이스에 user_id·session_id를 붙이고, 사람 주석 점수와 세션 점수를 남겨줘"`

In [7]:
# STEP 4-C — user·session 추적 + 사람 주석(Human Annotation)
from langfuse import get_client, propagate_attributes
from langfuse.openai import openai as lf_openai   # 트레이싱되는 OpenAI 드롭인

try:
    lf = get_client()
    USER_ID    = "user-demo-01"
    SESSION_ID = "session-demo-2026-07-02"
    QUESTION   = "강남 2만원 이하 매운 점심"

    # 1) user·session을 붙여 호출 → 트레이스가 '누가 / 어느 대화'로 묶인다
    with propagate_attributes(user_id=USER_ID, session_id=SESSION_ID):
        with lf.start_as_current_observation(name="맛집추천-데모", input=QUESTION) as span:
            resp = lf_openai.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": QUESTION}])
            answer = resp.choices[0].message.content.strip()
            trace_id = lf.get_current_trace_id()
    print("답변:", answer[:60], "…")
    print("  trace:", trace_id, "| user:", USER_ID, "| session:", SESSION_ID)

    # 2) 사람 주석(HITL) — 검수자가 그 트레이스에 👍/👎·등급을 남긴다
    lf.create_score(trace_id=trace_id, name="human_thumb",
                    value=1, data_type="BOOLEAN", comment="👍 좋은 추천 (사람 검수)")
    lf.create_score(trace_id=trace_id, name="human_rating",
                    value="good", data_type="CATEGORICAL", comment="good / ok / bad 중 good")

    # 3) 세션 단위 점수 — 대화(세션) 전체 만족도 (세션 점수는 NUMERIC/BOOLEAN)
    lf.create_score(session_id=SESSION_ID, name="session_satisfaction",
                    value=0.9, data_type="NUMERIC", comment="세션 전체 만족도")

    lf.flush()
    print("✅ 사람 주석 2개(trace) + 세션 점수 1개 기록 —")
    print("   UI ▸ Tracing ▸ Users / Sessions, 그리고 Scores에서 확인")
    print("   · 대규모 사람 검수는 UI의 'Annotation Queues'로 팀에 배분해 라벨링합니다.")
except Exception as e:
    print("건너뜀:", type(e).__name__, "-", e)
    print("· STEP 1(OPENAI_API_KEY)과 STEP 4(LANGFUSE 키/호스트)를 먼저 실행했는지 확인하세요.")

2026-07-10 23:21:12,610 - langfuse - DEBUG - Propagated 2 attributes to span '04ccc332193ab6a6': {'user.id': 'user-demo-01', 'session.id': 'session-demo-2026-07-02'}
2026-07-10 23:21:12,618 - langfuse - DEBUG - Propagated 2 attributes to span 'fb1b20c15b63a361': {'user.id': 'user-demo-01', 'session.id': 'session-demo-2026-07-02'}
2026-07-10 23:21:14,067 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "13f9a8ceeaf44016afaa55b067845c44",
    "span_id": "fb1b20c15b63a361",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": "04ccc332193ab6a6",
  "start_time": "2026-07-10T14:21:12.618319Z",
  "end_time": "2026-07-10T14:21:14.066986Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "user.id": "user-demo-01",
    "session.id": "session-demo-2026-07-02",
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\

답변: 강남의 "매운짬뽕" 추천합니다. 가격대가 2만원 이하이며, 매운 맛을 좋아하는 분들에게 적합합니다. …
  trace: 13f9a8ceeaf44016afaa55b067845c44 | user: user-demo-01 | session: session-demo-2026-07-02


2026-07-10 23:21:14,930 - langfuse - DEBUG - API: Uploading batch of 3 score events to Langfuse API
2026-07-10 23:21:14,933 - langfuse - DEBUG - making request: {"batch": [{"id": "9573e28af7157a0c1ef21c715df75ff2", "type": "score-create", "timestamp": "2026-07-10T14:21:14.070300Z", "body": {"id": "193e62f6bb14083d", "name": "human_thumb", "value": 1.0, "comment": "\ud83d\udc4d \uc88b\uc740 \ucd94\ucc9c (\uc0ac\ub78c \uac80\uc218)", "traceId": "13f9a8ceeaf44016afaa55b067845c44", "dataType": "BOOLEAN"}}, {"id": "93541ac65441aad9e3201323bd29168f", "type": "score-create", "timestamp": "2026-07-10T14:21:14.070738Z", "body": {"id": "3627d830d977f716", "name": "human_rating", "value": "good", "comment": "good / ok / bad \uc911 good", "traceId": "13f9a8ceeaf44016afaa55b067845c44", "dataType": "CATEGORICAL"}}, {"id": "026a006c579ae0d1f01130bfa8afa173", "type": "score-create", "timestamp": "2026-07-10T14:21:14.071454Z", "body": {"id": "89e9b576d2962150", "name": "session_satisfaction", "value": 

✅ 사람 주석 2개(trace) + 세션 점수 1개 기록 —
   UI ▸ Tracing ▸ Users / Sessions, 그리고 Scores에서 확인
   · 대규모 사람 검수는 UI의 'Annotation Queues'로 팀에 배분해 라벨링합니다.


## STEP 4-D — Annotation Queue 생성·분배·라벨링

사람 검수를 **팀 단위로 체계화**하는 기능입니다. 검수할 트레이스를 **큐(Queue)** 에 모아 **검수자에게 분배**하고,
검수자는 정해진 **라벨(Score Config)** 로 일관되게 채점합니다.

- **Score Config**: 라벨 스키마 정의(예: `good / ok / bad`) — 큐가 이 기준으로 채점되게 함
- **Queue 생성 + 항목 추가**: 검수 대상 트레이스를 큐에 넣음(`objectType="TRACE"`, 상태 `PENDING`)
- **분배(Assignment)**: 큐를 특정 검수자(`userId`)에게 할당 → UI에서 각자 자기 큐를 검수
- **라벨링**: 검수자가 라벨을 제출하면 **Score**가 생기고, 항목을 `COMPLETED`로 마감

> `userId`는 프로젝트 **멤버의 사용자 ID**여야 합니다(설정 ▸ Members). 실습에서는 `ANNOTATOR_USER_ID`가 비어 있으면 분배는 건너뜁니다.
> 실제 라벨링은 보통 **UI의 큐 화면**에서 사람이 하지만, 여기서는 API로 라벨 제출까지 시연합니다.

⌨️ **터미널 Codex:** `codex "사람 검수용 라벨(good/ok/bad)로 Annotation Queue를 만들고 트레이스를 넣어 검수자에게 분배해줘"`

In [8]:
# STEP 4-D — Annotation Queue: 라벨 정의 → 큐 생성 → 항목 추가 → 분배 → 라벨링 → 마감
import requests
from langfuse import get_client, propagate_attributes
from langfuse.openai import openai as lf_openai

def _lf_cfg2():
    host = os.environ.get("LANGFUSE_HOST", "").rstrip("/")
    pk, sk = os.environ.get("LANGFUSE_PUBLIC_KEY"), os.environ.get("LANGFUSE_SECRET_KEY")
    if not (host and pk and sk):
        raise RuntimeError("LANGFUSE_HOST/PUBLIC_KEY/SECRET_KEY 필요 (STEP 4 먼저 실행).")
    return host, (pk, sk)

def lf_api(method, path, payload=None):
    host, auth = _lf_cfg2()
    r = requests.request(method, f"{host}{path}", auth=auth, json=payload, timeout=60)
    if not r.ok:
        print(f"⚠️ {method} {path} -> {r.status_code}: {r.text}")
    r.raise_for_status()
    return r.json() if r.text else {}

ANNOTATOR_USER_ID = "cmr1r8zgw0007tx070roo4fzi"   # ← 분배 대상 검수자. 설정 ▸ Members의 사용자 ID. 비우면 분배 건너뜀.

try:
    lf = get_client()

    # 1) Score Config — 검수 라벨 스키마(good/ok/bad). 이미 있으면 재사용
    CFG_NAME = "추천품질-사람검수"
    try:
        cfg = lf_api("POST", "/api/public/score-configs", {
            "name": CFG_NAME,
            "dataType": "CATEGORICAL",
            "categories": [{"value": 1, "label": "good"},
                           {"value": 0.5, "label": "ok"},
                           {"value": 0, "label": "bad"}],
            "description": "사람 검수용 추천 품질 라벨",
        })
        cfg_id = cfg.get("id")
    except Exception:
        data = lf_api("GET", "/api/public/score-configs")
        data = data.get("data", data)
        cfg_id = next((c["id"] for c in data if c.get("name") == CFG_NAME), None)
    print("Score Config:", cfg_id)

    # 2) 검수 대상 트레이스 확보 — STEP 4-C의 trace_id 재사용, 없으면 새로 하나 생성
    oid = globals().get("trace_id")
    if not oid:
        with propagate_attributes(user_id="user-demo-01", session_id="session-demo-2026-07-02"):
            with lf.start_as_current_observation(name="검수대상-데모", input="강남 매운 점심") as _s:
                lf_openai.chat.completions.create(
                    model=MODEL, messages=[{"role": "user", "content": "강남 2만원 이하 매운 점심"}])
                oid = lf.get_current_trace_id()
        lf.flush()

    # 3) Queue 생성 (이 Score Config로 채점) — 이미 있으면 재사용
    Q_NAME = "맛집추천 검수 큐"
    try:
        q = lf_api("POST", "/api/public/annotation-queues", {
            "name": Q_NAME,
            "description": "사람 검수 대상 트레이스 모음",
            "scoreConfigIds": [cfg_id],
        })
        queue_id = q.get("id")
    except Exception:
        data = lf_api("GET", "/api/public/annotation-queues")
        data = data.get("data", data)
        queue_id = next((x["id"] for x in data if x.get("name") == Q_NAME), None)
    print("Queue:", queue_id)

    # 4) 큐에 항목 추가 — 검수할 트레이스를 PENDING 상태로
    item = lf_api("POST", f"/api/public/annotation-queues/{queue_id}/items", {
        "objectId": oid, "objectType": "TRACE", "status": "PENDING"})
    item_id = item.get("id")
    print("Queue Item:", item_id, "(trace:", oid, ")")

    # 5) 분배(Assignment) — 큐를 검수자에게 할당 (userId는 프로젝트 멤버 ID)
    if ANNOTATOR_USER_ID:
        try:
            lf_api("POST", f"/api/public/annotation-queues/{queue_id}/assignments",
                   {"userId": ANNOTATOR_USER_ID})
            print("✅ 분배 완료 → 검수자:", ANNOTATOR_USER_ID)
        except Exception as e:
            print("분배 건너뜀:", type(e).__name__, "— userId가 프로젝트 멤버인지 확인")
    else:
        print("ℹ️ ANNOTATOR_USER_ID 비어 있음 — 분배 생략 (설정 ▸ Members에서 ID 확인 후 채우세요)")

    # 6) 라벨링 — 검수자가 라벨 제출(보통 UI에서). API로는 Score로 기록 + 항목 COMPLETED 마감
    lf.create_score(trace_id=oid, name=CFG_NAME, value="good",
                    data_type="CATEGORICAL", config_id=cfg_id, comment="사람 검수: good")
    lf.flush()
    lf_api("PATCH", f"/api/public/annotation-queues/{queue_id}/items/{item_id}",
           {"status": "COMPLETED"})
    print("✅ 라벨 'good' 기록 + 항목 COMPLETED — UI ▸ Annotation Queues에서 확인")
except Exception as e:
    print("건너뜀:", type(e).__name__, "-", e)
    print("· STEP 4(Langfuse 키)와 STEP 1(OPENAI_API_KEY)을 먼저 실행했는지 확인하세요.")

2026-07-10 23:21:15,165 - langfuse - DEBUG - Score: Enqueuing event type=score-create for trace_id=None name=추천품질-사람검수 value=good
2026-07-10 23:21:15,165 - langfuse - DEBUG - Successfully flushed OTEL tracer provider


Score Config: e1c4469d-5143-4bd3-b609-7335aeef7f71
Queue: cmrf0x7ec000jqo07ebp9h02c
Queue Item: cmrf0x7g6000mqo07du30faws (trace: 13f9a8ceeaf44016afaa55b067845c44 )
⚠️ POST /api/public/annotation-queues/cmrf0x7ec000jqo07ebp9h02c/assignments -> 404: {"message":"User not found or not authorized for this project","error":"LangfuseNotFoundError"}
분배 건너뜀: HTTPError — userId가 프로젝트 멤버인지 확인


2026-07-10 23:21:15,984 - langfuse - DEBUG - API: Uploading batch of 1 score events to Langfuse API
2026-07-10 23:21:15,988 - langfuse - DEBUG - making request: {"batch": [{"id": "666edbdc007d4c44ca61aa6110139987", "type": "score-create", "timestamp": "2026-07-10T14:21:15.165037Z", "body": {"id": "abb19da18715cbeb", "name": "\ucd94\ucc9c\ud488\uc9c8-\uc0ac\ub78c\uac80\uc218", "value": "good", "comment": "\uc0ac\ub78c \uac80\uc218: good", "traceId": "13f9a8ceeaf44016afaa55b067845c44", "dataType": "CATEGORICAL", "configId": "e1c4469d-5143-4bd3-b609-7335aeef7f71"}}], "metadata": {"batch_size": 1, "sdk_name": "python", "sdk_version": "4.12.0", "public_key": "pk-lf-ecd84f7b-b38a-4be5-9458-a40eb8a03784"}} to http://localhost:3000/api/public/ingestion
2026-07-10 23:21:16,028 - langfuse - DEBUG - received response: {"successes":[{"id":"666edbdc007d4c44ca61aa6110139987","status":201}],"errors":[]}
2026-07-10 23:21:16,028 - langfuse - DEBUG - API: Successfully sent 1 score events to Langfuse API

✅ 라벨 'good' 기록 + 항목 COMPLETED — UI ▸ Annotation Queues에서 확인


## STEP 5 — 심판의 위치 편향 실험

강의의 심판 편향 — 같은 두 답을 **순서만 바꿔** 비교시키면 판정이 뒤집힐 수 있습니다. 직접 확인합니다.

In [9]:
q = DATASET[0]["input"]
ans_a = app(q, temperature=0.2)
ans_b = app(q, temperature=1.2)
def prefer(first, second):
    r = client.chat.completions.create(model=MODEL, temperature=0, messages=[
        {"role":"system","content":"두 답 중 더 나은 쪽을 'A' 또는 'B' 한 글자로만 답하라."},
        {"role":"user","content":f"질문: {q}\n\nA: {first}\n\nB: {second}"}])
    return r.choices[0].message.content.strip()[:1]
v1 = prefer(ans_a, ans_b)          # A=저온, B=고온
v2 = prefer(ans_b, ans_a)          # 순서 뒤집기 — A=고온, B=저온
consistent = (v1 == "A" and v2 == "B") or (v1 == "B" and v2 == "A")
print(f"원래 순서 판정: {v1} / 뒤집은 순서 판정: {v2}")
print("✅ 일관 — 위치 편향 없음" if consistent else "⚠ 판정이 뒤집혔다 — 위치 편향! 순서 랜덤화 + 2회 채점으로 완화")

2026-07-10 23:21:17,432 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "b22c729a3b323045a1bfa795952e68ec",
    "span_id": "d5e6b61f6e100a44",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T14:21:16.066937Z",
  "end_time": "2026-07-10T14:21:17.432658Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8\\ub2e4. \\uc0ac\\uc6a9\\uc790 \\uc694\\uccad\\uc5d0 \\ud55c\\uad6d\\uc5b4\\ub85c \\uac04\\uacb0\\ud788, \\ud55c \\uacf3\\ub9cc \\ucd94\\ucc9c\\ud558\\ub77c.\"}, {\"role\": \"user\", \"content\": \"\\uac15\\ub0a8\\uc5d0\\uc11c 2\\ub9cc\\uc6d0 \\uc774\\ud558 \\ub9e4\\uc6b4 \\uc810\\uc2ec \\ucd94\\ucc9c\\ud574\\uc918.\"}]",
    "langfus

원래 순서 판정: A / 뒤집은 순서 판정: B
✅ 일관 — 위치 편향 없음


## STEP 6 — 선호 비교(pairwise)로 승률 내기

절대 점수보다 '어느 쪽이 나은가'가 안정적입니다. 두 설정의 **승률**을 냅니다 (Chatbot Arena의 미니어처).

In [14]:
wins = {"low_temp": 0, "high_temp": 0}
for item in DATASET[:3]:
    a = app(item["input"], temperature=0.2)
    b = app(item["input"], temperature=1.2)
    v = prefer(a, b)                     # STEP 5의 심판 재사용
    wins["low_temp" if v == "A" else "high_temp"] += 1
    print(f"  {item['input'][:30]}... → {'저온 승' if v=='A' else '고온 승'}")
print(f"\n승률: 저온 {wins['low_temp']}/3 vs 고온 {wins['high_temp']}/3 — 이긴 설정을 production으로")
# 실전: 순서 랜덤화 + 심판 앙상블로 편향을 더 줄인다

2026-07-11 01:23:59,340 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "dd58d78f2b5d25981c274e17b9db0bf2",
    "span_id": "d3988986db3069ce",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T16:23:57.222012Z",
  "end_time": "2026-07-10T16:23:59.339899Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8\\ub2e4. \\uc0ac\\uc6a9\\uc790 \\uc694\\uccad\\uc5d0 \\ud55c\\uad6d\\uc5b4\\ub85c \\uac04\\uacb0\\ud788, \\ud55c \\uacf3\\ub9cc \\ucd94\\ucc9c\\ud558\\ub77c.\"}, {\"role\": \"user\", \"content\": \"\\uac15\\ub0a8\\uc5d0\\uc11c 2\\ub9cc\\uc6d0 \\uc774\\ud558 \\ub9e4\\uc6b4 \\uc810\\uc2ec \\ucd94\\ucc9c\\ud574\\uc918.\"}]",
    "langfus

  강남에서 2만원 이하 매운 점심 추천해줘.... → 저온 승


2026-07-11 01:24:03,416 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "50fe80c62a21ad28eba7224e9302cfc8",
    "span_id": "265928230ce29ed0",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T16:24:01.778311Z",
  "end_time": "2026-07-10T16:24:03.416599Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8\\ub2e4. \\uc0ac\\uc6a9\\uc790 \\uc694\\uccad\\uc5d0 \\ud55c\\uad6d\\uc5b4\\ub85c \\uac04\\uacb0\\ud788, \\ud55c \\uacf3\\ub9cc \\ucd94\\ucc9c\\ud558\\ub77c.\"}, {\"role\": \"user\", \"content\": \"\\ud63c\\ubc25\\ud558\\uae30 \\uc88b\\uc740 \\uad6d\\ubc25\\uc9d1 \\ud55c \\uacf3\\ub9cc \\ucd94\\ucc9c\\ud574\\uc918.\"}]",
    "langfuse

  혼밥하기 좋은 국밥집 한 곳만 추천해줘.... → 저온 승


2026-07-11 01:24:06,773 - langfuse - DEBUG - Trace: Processing span name='OpenAI-generation' | Full details:
{
  "name": "OpenAI-generation",
  "context": {
    "trace_id": "2e9a8491cef668141d35c331307b3773",
    "span_id": "4d9dd8602ac03e98",
    "trace_state": "[]"
  },
  "kind": "SpanKind.INTERNAL",
  "parent_id": null,
  "start_time": "2026-07-10T16:24:05.584635Z",
  "end_time": "2026-07-10T16:24:06.773718Z",
  "status": {
    "status_code": "UNSET"
  },
  "attributes": {
    "langfuse.internal.is_app_root": true,
    "langfuse.observation.input": "[{\"role\": \"system\", \"content\": \"\\ub108\\ub294 \\ub9db\\uc9d1 \\ucd94\\ucc9c \\ub3c4\\uc6b0\\ubbf8\\ub2e4. \\uc0ac\\uc6a9\\uc790 \\uc694\\uccad\\uc5d0 \\ud55c\\uad6d\\uc5b4\\ub85c \\uac04\\uacb0\\ud788, \\ud55c \\uacf3\\ub9cc \\ucd94\\ucc9c\\ud558\\ub77c.\"}, {\"role\": \"user\", \"content\": \"\\ud64d\\ub300 \\uadfc\\ucc98 \\ub370\\uc774\\ud2b8\\ud558\\uae30 \\uc88b\\uc740 \\ubd84\\uc704\\uae30 \\uc2dd\\ub2f9 \\uc54c\\ub824\\uc91

  홍대 근처 데이트하기 좋은 분위기 식당 알려줘.... → 저온 승

승률: 저온 3/3 vs 고온 0/3 — 이긴 설정을 production으로


## STEP 7 — ⭐ 내 MVP에 적용

당신 팀 MVP의 **평가 셋**을 만들고 채점 기준을 정하세요.
1. 대표 입력 5~10개를 `MY_DATASET`에 모은다(어려운/실패했던 케이스 위주).
2. 우리 도메인에 맞는 **규칙 평가자**와 **LLM 심판 기준(rubric)** 을 정한다.
3. 프롬프트/모델을 바꿀 때마다 **같은 셋으로 실험** → 평균 점수로 회귀를 잡는다.

⌨️ **터미널 Codex:** `codex "우리 MVP 평가 데이터셋과 규칙·LLM심판 평가자를 만들고 실험으로 비교해줘"`

In [11]:
# 팀별 작성: 우리 MVP 평가 데이터셋 (대표 입력 + 기대 핵심)
MY_DATASET = [
    # {"input": "우리 서비스에 올 법한 요청", "expected": "반드시 포함할 핵심 또는 None"},
]
if MY_DATASET:
    r = run_experiment(MY_DATASET, "MY·baseline", temperature=0.7)
    print("\n→ 저녁 팀 프로젝트: 이 평가 셋을 Langfuse Datasets에 올리고, 프롬프트 버전마다 run_experiment로 비교하세요.")
else:
    print("⬜ MY_DATASET을 채운 뒤 다시 실행하세요 — 우리 MVP의 첫 '평가 셋 + 점수'")

⬜ MY_DATASET을 채운 뒤 다시 실행하세요 — 우리 MVP의 첫 '평가 셋 + 점수'


## 정리

- **평가 = 데이터셋 × 평가자 → 점수(Score).** 느낌이 아니라 숫자로 "더 나은가"를 답한다.
- 평가자 두 축: **규칙 기반**(싸고 결정적) + **LLM-as-a-judge**(사람에 가깝게, 이유 포함).
- Langfuse 평가: `create_dataset`·`create_dataset_item` → `dataset.run_experiment(task, evaluators)` → **Runs/Scores**에서 버전 비교.
- **두 가지 평가자 방식**: ① 코드 evaluator(`evaluators=[...]`, `Evaluation` 반환) ② **Langfuse에 Evaluator 등록**(unstable API) → 서버가 자동 채점(팀 공유·상시 재사용).
- 19차시 프롬프트 버전을 이 실험으로 **객관적으로 비교**해 `production`으로 승격 — 프롬프트 관리와 평가가 한 루프.
- 상시 평가(카나리아)로 **회귀를 사용자보다 먼저** 잡는다(18차시 관찰성과 연결).
- **사람 + 맥락**: `user_id`/`session_id`로 사용자·세션별 품질을 묶고, **사람 주석(Score)**·Annotation Queues로 HITL을 자동 평가와 한 뷰에서 비교.
- **Annotation Queue**: 라벨 스키마(Score Config)로 큐를 만들어 검수 대상 트레이스를 **검수자에게 분배**하고 일관되게 **라벨링** → 팀 단위 사람 평가 파이프라인.
- 오늘 만든 평가 셋은 **저녁 팀 프로젝트**에 그대로. 다음 차시(21): 안전·거버넌스.